# production registry health v1

目标：验证 `production/production_registry.json` 的 current 指针能通过稳定 `src.schedule` 接口生成可给 dashboard/schedule 使用的健康检查 export。

边界：notebook 只编排证据，不直接实现路径检查逻辑；路径检查逻辑必须在 `src.schedule`。


In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.schedule import ProductionHealthCheckConfig, run_production_health_check_once

CARD = Path('research/production_registry_health_v1')
CARD.mkdir(parents=True, exist_ok=True)
events = []
config = ProductionHealthCheckConfig(
    name='research_production_health_check_v1',
    registry_path=Path('production/production_registry.json'),
    strategy_id='paper_sim_replay_v1',
    output_path=CARD / 'production_health_export.json',
    notify_details=True,
)
config


ProductionHealthCheckConfig(name='research_production_health_check_v1', registry_path=PosixPath('production/production_registry.json'), strategy_id='paper_sim_replay_v1', output_path=PosixPath('research/production_registry_health_v1/production_health_export.json'), required_paths=('notebook', 'summary', 'dashboard_export', 'intent_ledger', 'paper_execution_result', 'data_registry', 'analysis_manifest'), require_requested_model_contract=False, notify_details=True, retries=0, market_calendar_decision=None, notify_skipped=False)

In [2]:
result = run_production_health_check_once(config, lock_dir=CARD / 'locks', notify=events.append)
result


JobResult(name='research_production_health_check_v1', status='success', payload={'schema_version': 'schedule.production_health.v1', 'strategy_id': 'paper_sim_replay_v1', 'health_status': 'healthy', 'is_healthy': True, 'registry_path': 'production/production_registry.json', 'manifest_path': 'production/paper_sim_replay_v1/production_manifest.json', 'manifest_hash': 'a5c23f8aedf86b04eac9576231f3bd659f8ef7017abb0b3a83340693e03b989c', 'registry_manifest_hash': 'a5c23f8aedf86b04eac9576231f3bd659f8ef7017abb0b3a83340693e03b989c', 'computed_manifest_hash': 'a5c23f8aedf86b04eac9576231f3bd659f8ef7017abb0b3a83340693e03b989c', 'manifest_hash_matches_registry': True, 'manifest_hash_matches_content': True, 'artifact_id': 'paper_sim_replay_v1', 'artifact_type': 'production_strategy', 'mode': 'paper', 'status': 'paper_replay_completed', 'live_broker_enabled': False, 'paper_execution_pipeline_schema_version': 'schedule.paper_execution.v1', 'paper_execution_job_status': 'success', 'published_at': '2026-

In [3]:
health_export = json.loads((CARD / 'production_health_export.json').read_text(encoding='utf-8'))
summary = {
    'schema_version': 'research.production_registry_health.v1',
    'status': result.status,
    'strategy_id': health_export['strategy_id'],
    'health_status': health_export['health_status'],
    'is_healthy': health_export['is_healthy'],
    'manifest_hash': health_export['manifest_hash'],
    'manifest_path': health_export['manifest_path'],
    'mode': health_export['mode'],
    'live_broker_enabled': health_export['live_broker_enabled'],
    'checked_path_count': health_export['checked_path_count'],
    'missing_required_paths': health_export['missing_required_paths'],
    'dashboard_export_path': health_export['dashboard_export_path'],
    'notification': events[0] if events else '',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


{'schema_version': 'research.production_registry_health.v1',
 'status': 'success',
 'strategy_id': 'paper_sim_replay_v1',
 'health_status': 'healthy',
 'is_healthy': True,
 'manifest_hash': 'a5c23f8aedf86b04eac9576231f3bd659f8ef7017abb0b3a83340693e03b989c',
 'manifest_path': 'production/paper_sim_replay_v1/production_manifest.json',
 'mode': 'paper',
 'live_broker_enabled': False,
 'checked_path_count': 7,
 'missing_required_paths': [],
 'dashboard_export_path': 'production/paper_sim_replay_v1/dashboard_export.json',
 'notification': 'research_production_health_check_v1:success | strategy=paper_sim_replay_v1 | health=healthy | missing=0 | paper_execution=schedule.paper_execution.v1/success | requested_models=2/32 missing=30 incomplete=0 | manifest=production/paper_sim_replay_v1/production_manifest.json'}

In [4]:
assert result.status == 'success'
assert health_export['schema_version'] == 'schedule.production_health.v1'
assert health_export['strategy_id'] == 'paper_sim_replay_v1'
assert health_export['health_status'] == 'healthy'
assert health_export['is_healthy'] is True
assert health_export['mode'] == 'paper'
assert health_export['live_broker_enabled'] is False
assert health_export['missing_required_paths'] == []
assert health_export['dashboard_export_path'].endswith('dashboard_export.json')
assert 'health=healthy' in summary['notification']
summary


{'schema_version': 'research.production_registry_health.v1',
 'status': 'success',
 'strategy_id': 'paper_sim_replay_v1',
 'health_status': 'healthy',
 'is_healthy': True,
 'manifest_hash': 'a5c23f8aedf86b04eac9576231f3bd659f8ef7017abb0b3a83340693e03b989c',
 'manifest_path': 'production/paper_sim_replay_v1/production_manifest.json',
 'mode': 'paper',
 'live_broker_enabled': False,
 'checked_path_count': 7,
 'missing_required_paths': [],
 'dashboard_export_path': 'production/paper_sim_replay_v1/dashboard_export.json',
 'notification': 'research_production_health_check_v1:success | strategy=paper_sim_replay_v1 | health=healthy | missing=0 | paper_execution=schedule.paper_execution.v1/success | requested_models=2/32 missing=30 incomplete=0 | manifest=production/paper_sim_replay_v1/production_manifest.json'}

## 中文结论

- 做了什么：使用 `src.schedule.run_production_health_check_once` 读取 `production/production_registry.json` 的 `paper_sim_replay_v1` current 指针，检查 production manifest 中声明的 notebook、summary、dashboard_export、intent_ledger、paper_execution_result、data_registry 和 analysis_manifest 是否存在，并写出 `production_health_export.json`。
- 核心指标：`health_status=healthy`，`is_healthy=True`，`checked_path_count=7`，`missing_required_paths=[]`，manifest hash 与 registry current 一致。
- regime 差异：本 notebook 不重新计算 regime；它验证 production manifest 已把 dashboard/analysis 产物作为健康检查输入，regime 差异仍由 dashboard export 和 analysis manifest 承载。
- 是否晋升：否。本实验只验证 production registry 健康检查链路，不改变策略晋升状态。
- 风险：健康检查只能证明 manifest 声明的 sidecar 存在且 hash/current 指针一致，不能证明模型未来表现或真实 broker 可用；v1 仍保持 paper/sim。
- 下一步：把 `production_health_export.json` 接入 dashboard 页面或外部定时通知，让 production current 指针异常时能直接暴露给 review。
